Installing Dependencies

In [2]:
!pip install tensorflow sentencepiece wikipedia-api

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia-api: filename=Wikipedia_API-0.8.1-py3-none-any.whl size=15383 sha256=29b1e0cf7db03e86b60fc44a018b1ca536749bce17185db9ddd3f8969100f7bc
  Stored in directory: /root/.cache/pip/wheels/33/3c/79/b36253689d838af4a0539782853ac3cc38a83a6591ad570dde
Successfully built wikipedia-api


Constants

In [11]:
SEQ_LEN = 256
BATCH_SIZE = 32
EMBED_DIM = 512
NUM_HEADS = 8
NUM_LAYERS = 6
MLP_RATIO = 4
DROPOUT = 0.1
EPOCHS = 7
LEARNING_RATE = 3e-4

Utility

In [4]:
import numpy as np
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

# ---------- Utility: causal mask ----------


def causal_attention_mask(batch_size, seq_len):
    i = tf.range(seq_len)[:, None]
    j = tf.range(seq_len)
    mask = tf.cast(i >= j, tf.int32)
    mask = tf.reshape(mask, (1, 1, seq_len, seq_len))
    return tf.tile(mask, [batch_size, 1, 1, 1])


# ---------- Sampling ----------


def top_k_logits(logits, k):
    values, _ = tf.math.top_k(logits, k=k)
    min_values = values[:, -1, tf.newaxis]
    return tf.where(logits < min_values, -1e10, logits)


def sample_sequence(model, seed_tokens, length, temperature=1.0, top_k=50):
    cur = np.array(seed_tokens, dtype=np.int32)[None, :]
    for _ in range(length):
        if cur.shape[1] > model.seq_len:
            cur = cur[:, -model.seq_len:]
        logits = model(cur, training=False)
        last_logits = logits[0, -1, :] / temperature
        last_logits = top_k_logits(last_logits[tf.newaxis, :], top_k)[0]
        probs = tf.nn.softmax(last_logits)
        next_id = tf.random.categorical(tf.math.log(
            probs)[tf.newaxis, :], 1)[0, 0].numpy()
        cur = np.concatenate([cur, [[next_id]]], axis=1)
    return cur[0].tolist()

# ---------- Create dataset ----------


def create_dataset(sp, file_path, seq_len=SEQ_LEN, batch_size=BATCH_SIZE):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    ids = sp.encode(text, out_type=int)
    print(f"Encoded {len(ids)} tokens from corpus.")

    n_windows = (len(ids) - 1) // seq_len
    inputs = []
    targets = []
    for i in range(n_windows):
        start = i * seq_len
        inputs.append(ids[start:start + seq_len])
        targets.append(ids[start + 1:start + 1 + seq_len])

    inputs = np.array(inputs, dtype=np.int32)
    targets = np.array(targets, dtype=np.int32)

    ds = tf.data.Dataset.from_tensor_slices((inputs, targets))
    ds = ds.shuffle(1000).batch(
        batch_size, drop_remainder=True).prefetch(AUTOTUNE)
    return ds

Mounting Drive and Loading Dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dataset_corpus_path = "/content/drive/MyDrive/MiniGPT/corpus.txt"
tokenizer_output_dir = "/content/drive/MyDrive/MiniGPT/tokenizer"

Prepare Dataset

In [5]:
import sentencepiece as spm

# Path to your text corpus (plain text)
input_file = dataset_corpus_path

model_prefix = f"{tokenizer_output_dir}/spm"

# Train a SentencePiece model
spm.SentencePieceTrainer.train(
    input=input_file,
    model_prefix=model_prefix,
    vocab_size=16000,
    character_coverage=1.0,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

print("✅ SentencePiece model trained: spm.model, spm.vocab")


✅ SentencePiece model trained: spm.model, spm.vocab


Models

In [8]:
import tensorflow as tf


class CausalSelfAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim//num_heads)
        self.proj = tf.keras.layers.Dense(embed_dim)
        self.dropout = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=None, mask=None):
        attn_mask = tf.cast(tf.squeeze(mask, axis=1),
                            tf.bool) if mask is not None else None
        y = self.mha(query=x, value=x, key=x,
                     attention_mask=attn_mask, training=training)
        y = self.dropout(self.proj(y), training=training)
        return y


class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_ratio, dropout):
        super().__init__()
        self.ln1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-5, dtype="float32")
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.ln2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-5, dtype="float32")
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(
                embed_dim * mlp_ratio, activation=tf.keras.activations.gelu, dtype="float32"),
            tf.keras.layers.Dense(embed_dim, dtype="float32"),
            tf.keras.layers.Dropout(dropout)
        ])

    def call(self, x, training=None, mask=None):
        x = x + self.attn(self.ln1(x), training=training, mask=mask)
        x = x + self.mlp(self.ln2(x), training=training)
        return x


MiniGPT

In [9]:
from tensorflow.keras import mixed_precision
import tensorflow as tf
import keras

# ---------- Mixed precision ----------
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision enabled:", mixed_precision.global_policy())


@keras.saving.register_keras_serializable(package="CustomModels")
class MiniGPT(tf.keras.Model):
    def __init__(self, vocab_size, seq_len, embed_dim, num_heads, num_layers, mlp_ratio, dropout, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len
        self.token_emb = tf.keras.layers.Embedding(
            vocab_size, embed_dim, dtype="float32")
        self.pos_emb = tf.Variable(
            tf.zeros([1, seq_len, embed_dim], dtype=tf.float32), trainable=True)
        self.drop = tf.keras.layers.Dropout(dropout)
        self.blocks = [TransformerBlock(
            embed_dim, num_heads, mlp_ratio, dropout) for _ in range(num_layers)]
        self.ln_f = tf.keras.layers.LayerNormalization(
            epsilon=1e-5, dtype="float32")
        self.head = tf.keras.layers.Dense(vocab_size, dtype="float32")

    def call(self, idx, training=None):
        B, T = tf.shape(idx)[0], tf.shape(idx)[1]
        tok = self.token_emb(idx)
        x = tok + self.pos_emb[:, :T, :]
        x = self.drop(x, training=training)
        mask = causal_attention_mask(B, T)
        for block in self.blocks:
            x = block(x, training=training, mask=mask)
        x = self.ln_f(x)
        return self.head(x)


Mixed precision enabled: <DTypePolicy "mixed_float16">


Training

In [12]:
import os
import tensorflow as tf
import sentencepiece as spm


# ---------- Load SentencePiece tokenizer ----------
SP_MODEL = "spm.model"
sp = spm.SentencePieceProcessor()
sp.load(SP_MODEL)
VOCAB_SIZE = sp.get_piece_size()
print(f"✅ Loaded SentencePiece model with vocab size: {VOCAB_SIZE}")

CHECKPOINT_DIR = "/content/drive/MyDrive/MiniGPT"


def train_model():
    dataset = create_dataset(sp, dataset_corpus_path)

    model = MiniGPT(VOCAB_SIZE, SEQ_LEN, EMBED_DIM,
                    NUM_HEADS, NUM_LAYERS, MLP_RATIO, DROPOUT)
    opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            logits = model(x, training=True)
            loss = loss_fn(y, logits)
        grads = tape.gradient(loss, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for epoch in range(EPOCHS):
        for step, (x, y) in enumerate(dataset):
            print("$%$%$", step)
            loss = train_step(x, y)
            if step % 10 == 0:
                print(f"Epoch {epoch} Step {step} Loss {loss.numpy():.4f}")

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    model.save_weights(os.path.join(
        CHECKPOINT_DIR, "minigpt_full_model.weights.h5"))
    print("✅ Training completed and model saved.")

    # Sampling
    seed_text = "An algorithm is"
    seed_ids = sp.encode(seed_text, out_type=int)
    generated_ids = sample_sequence(model, seed_ids, 50)
    print("🪄 Generated:", sp.decode(generated_ids))


if __name__ == "__main__":
    train_model()


✅ Loaded SentencePiece model with vocab size: 8000
Encoded 720296 tokens from corpus.
$%$%$ 0
Epoch 0 Step 0 Loss 9.0251
$%$%$ 1
$%$%$ 2
$%$%$ 3
$%$%$ 4
$%$%$ 5
$%$%$ 6
$%$%$ 7
$%$%$ 8
$%$%$ 9
$%$%$ 10
Epoch 0 Step 10 Loss 8.3030
$%$%$ 11
$%$%$ 12
$%$%$ 13
$%$%$ 14
$%$%$ 15
$%$%$ 16
$%$%$ 17
$%$%$ 18
$%$%$ 19
$%$%$ 20
Epoch 0 Step 20 Loss 7.8080
$%$%$ 21
$%$%$ 22
$%$%$ 23
$%$%$ 24
$%$%$ 25
$%$%$ 26
$%$%$ 27
$%$%$ 28
$%$%$ 29
$%$%$ 30
Epoch 0 Step 30 Loss 7.5159
$%$%$ 31
$%$%$ 32
$%$%$ 33
$%$%$ 34
$%$%$ 35
$%$%$ 36
$%$%$ 37
$%$%$ 38
$%$%$ 39
$%$%$ 40
Epoch 0 Step 40 Loss 7.2825
$%$%$ 41
$%$%$ 42
$%$%$ 43
$%$%$ 44
$%$%$ 45
$%$%$ 46
$%$%$ 47
$%$%$ 48
$%$%$ 49
$%$%$ 50
Epoch 0 Step 50 Loss 7.4766
$%$%$ 51
$%$%$ 52
$%$%$ 53
$%$%$ 54
$%$%$ 55
$%$%$ 56
$%$%$ 57
$%$%$ 58
$%$%$ 59
$%$%$ 60
Epoch 0 Step 60 Loss 6.9342
$%$%$ 61
$%$%$ 62
$%$%$ 63
$%$%$ 64
$%$%$ 65
$%$%$ 66
$%$%$ 67
$%$%$ 68
$%$%$ 69
$%$%$ 70
Epoch 0 Step 70 Loss 7.0064
$%$%$ 71
$%$%$ 72
$%$%$ 73
$%$%$ 74
$%$%$ 75
$%$%$ 76
$%$%$ 77